# 04 · Baseline Popularity và item-kNN liền kề (ticket #9)

**Input** (Add Input): Kaggle Dataset `titeohihi/mixnext-splited` (output của notebook 03: `tracks.parquet`, `sessions.parquet`, `eval_targets.parquet`).

**Output** (`/kaggle/working/out/candidates/`): `popularity_{val,test}_run0.parquet`, `itemknn_{val,test}_run0.parquet`. Hai baseline tất định nên chỉ có lần chạy 0. Thư mục `out/` là nội dung của Kaggle Dataset `mixnext-candidates`.

Bật Internet để clone repo. Chỉ cần CPU.

In [ ]:
# Bootstrap: lấy code từ GitHub (#9 đã merge vào main).
BRANCH = "main"
!rm -rf /kaggle/working/MusicRecSys && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys /kaggle/working/MusicRecSys
import sys; sys.path.insert(0, "/kaggle/working/MusicRecSys")

In [ ]:
from pathlib import Path
DATA = sorted(Path("/kaggle/input").rglob("eval_targets.parquet"))[0].parent
OUT = Path("/kaggle/working/out")
print("input:", DATA, sorted(p.name for p in DATA.iterdir()))

In [ ]:
import time
from src.models.baselines import main

t0 = time.time()
report = main(["--data", str(DATA), "--out", str(OUT)])   # có validate_dataset bên trong; in HitRate@10, NDCG@10, MRR trên val
print(f"{time.time() - t0:.0f} giây")
for p in sorted((OUT / "candidates").iterdir()):
    print(p.name, f"{p.stat().st_size / 1e6:.1f} MB")

## Đăng candidates

1. **Save Version → Save & Run All (Commit)** để lưu output.
2. Ở tab Output của version vừa chạy → **New Dataset** tên `mixnext-candidates`. Các mô hình gu sau (#14–#16) thêm file của mình vào version mới của bộ này.

## Kết quả trên val (chạy trên Kaggle, CPU, 26/11/2026, 118 giây cho cả hai mô hình)

| mô hình | HitRate@10 | NDCG@10 | MRR | số phiên |
|---|---:|---:|---:|---:|
| popularity | 0,0089 | 0,0040 | 0,0037 | 9.212 |
| itemknn | 0,1921 | 0,1297 | 0,1165 | 9.212 |

MRR tính trên cả 100 ứng viên (`list_length` = 100).

- item-kNN liền kề hơn Popularity khoảng 20 lần về HitRate@10: bài kế tiếp phụ thuộc mạnh vào bài đang phát, đúng giả định của bài toán.
- Mốc cho ngưỡng "NDCG@10 giảm ≤ 10%" (spec §7): nếu xếp lại trên candidates item-kNN thì NDCG@10 phải còn ≥ 0,1168. Mô hình gu ở #14–#16 cần vượt NDCG@10 = 0,1297 thì mới đáng dùng thay item-kNN.